In [ ]:


import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

data = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')



In [ ]:
data.head()

In [ ]:
data = np.array(data)

In [ ]:
# print(data[:5])  # tampilkan 5 baris pertama


In [ ]:
m,n = data.shape
np.random.shuffle(data)

Lalu, m, n = data.shape artinya:
m menyimpan jumlah data (jumlah baris → 42000)

n menyimpan jumlah fitur + label (jumlah kolom → 785)

🔹 np.random.shuffle(data)
🔍 Fungsi:
Ini mengacak urutan baris dalam array data.

🧠 Penting: hanya baris yang diacak, isi per baris (fitur dan label) tetap satu paket dan tidak rusak.

📌 Kenapa ini penting?
Karena dataset dari Kaggle seperti "Digit Recognizer" biasanya urut:

Baris 0–4000 → angka 0

Baris 4001–8000 → angka 1

dst.

Kalau tidak diacak, maka model akan belajar dari pola yang tidak mewakili keseluruhan data. Misalnya:

Model hanya lihat angka 0 terus di awal, jadi berat sebelah.

Dengan np.random.shuffle(data), barisan data jadi acak:

Baris pertama bisa angka 7

Baris kedua angka 2

Baris ketiga angka 0

...dan seterusnya

In [ ]:
data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n]

data_train = data[1000 : m].T
Y_train = data_train[0]
X_train = data_train[1:n]

🧠 Penjelasan Pelan-Pelan:
✅ Tujuan Potongan Kode Ini:
Memisahkan dataset menjadi:

Data Latih (Training set) → Untuk melatih model

Data Uji (Dev/Test set) → Untuk menguji model

🔍 Penjelasan Baris per Baris:
1️⃣ data[0:1000].T
Kita ambil 1.000 baris pertama dari dataset (data gambar + label), lalu kita .T = Transpose (mengubah baris jadi kolom).

Kenapa di-transpose?

Karena nanti kita ingin data dalam bentuk:

Setiap kolom = 1 gambar

Setiap baris = nilai pixel ke-1, ke-2, dst

Misalnya:

plaintext
Sebelum Transpose:
  Baris 1: [Label, Pixel1, Pixel2, ..., Pixel784]
  Baris 2: [Label, Pixel1, Pixel2, ..., Pixel784]
  
Setelah Transpose:
  Baris 0: Label semua gambar
  Baris 1: Pixel pertama dari semua gambar
  Baris 2: Pixel kedua dari semua gambar

  
2️⃣ Y_dev = data_dev[0]
Baris ke-0 = Label-nya
Contoh:
Y_dev = [5, 0, 3, 9, 1, ...]
Berarti gambar ke-1 adalah angka 5, gambar ke-2 angka 0, dst.

3️⃣ X_dev = data_dev[1:n]
Baris ke-1 sampai ke-n = pixel-pixel dari semua gambar

Ukuran X_dev adalah: (784, 1000)

Artinya: 784 pixel per gambar, dan total ada 1000 gambar

4️⃣ data[1000 : m].T
Ambil sisa data dari baris ke-1000 sampai akhir untuk training. Transpose lagi biar bentuknya sama seperti data_dev.

5️⃣ Y_train = data_train[0]
Label semua data training.

6️⃣ X_train = data_train[1:n]
Pixel semua gambar training (sudah dipecah dari label-nya).

📦 Hasil Akhir:
X_train = Gambar untuk melatih model

Y_train = Label angka dari gambar itu

X_dev = Gambar untuk menguji model

Y_dev = Jawaban seharusnya dari gambar uji


Kenapa harus di transpose ?
banyak implementasi machine learning (terutama dalam numpy), kita lebih nyaman kalau setiap kolom adalah satu gambar, supaya:
- Setiap gambar jadi 1 kolom (bukan baris)
- Kita bisa proses banyak gambar sekaligus secara vectorized


In [ ]:
Y_train

In [ ]:
def init_params():
    W1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(Z, 0)

def softmax(Z):
    A = np.exp(Z) / sum(np.exp(Z))
    return A
    
def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def ReLU_deriv(Z):
    return Z > 0

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2)
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1)
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1    
    W2 = W2 - alpha * dW2  
    b2 = b2 - alpha * db2    
    return W1, b1, W2, b2

✅ 1. init_params()
python
Copy
Edit
def init_params():
    W1 = np.random.randn(10,784) - 0.5
    b1 = np.random.randn(10, 1) - 0.5
    W2 = np.random.randn(10,10) - 0.5
    b2 = np.random.randn(10, 1) - 0.5
    return W1, b1, W2, b2
Tujuan:
Inisialisasi bobot (W) dan bias (b) untuk dua layer jaringan saraf:

W1: bobot dari input (784 pixel) ke hidden layer (10 neuron).

b1: bias untuk hidden layer (10 neuron).

W2: bobot dari hidden layer (10 neuron) ke output layer (10 kelas → angka 0–9).

b2: bias untuk output layer.

Random kecil-kecil agar jaringan tidak mulai dari nol semua (tidak belajar).

✅ 2. ReLU(Z)
python
Copy
Edit
def ReLU(Z):
    return np.maximum(0, Z)
Tujuan:
Aktivasi ReLU (Rectified Linear Unit), dipakai di hidden layer untuk memperkenalkan non-linearity.
Jika nilai negatif → jadi 0, kalau positif → tetap.

✅ 3. softmax(Z)
python
Copy
Edit
def softmax(Z):
    return exp(Z) / np.sum(exp(Z))
Tujuan:
Mengubah output layer jadi probabilitas (jumlah total = 1).
Cocok untuk klasifikasi multi-kelas (angka 0–9).

📌 Catatan: kamu lupa np. di exp, harusnya:

python
Copy
Edit
return np.exp(Z) / np.sum(np.exp(Z), axis=0)
⚠️ 4. forward_prop(W1, b1, W2, b2)
python
Copy
Edit
def forward_prop(W1, b1, W2, b2):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot + b2
    A2 = softmax(A1)
Tujuan:
Forward propagation (maju) menghitung output dari input X.

TAPI ADA ERROR:

W2.dot + b2 seharusnya:

python
Copy
Edit
Z2 = W2.dot(A1) + b2
softmax(A1) salah, seharusnya:

python
Copy
Edit
A2 = softmax(Z2)
Dan function ini belum return apa pun, harusnya:

python
Copy
Edit
return Z1, A1, Z2, A2
✅ 5. one_hot(Y)
python
Copy
Edit
def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y
Tujuan:
Mengubah label angka (misal 5) jadi vektor seperti [0,0,0,0,0,1,0,0,0,0].

📌 Typo:

python
Copy
Edit
one_hoy_Y = one_hot_Y.T
return one_hot_Y
Harusnya:

python
Copy
Edit
one_hot_Y = one_hot_Y.T
return one_hot_Y
✅ 6. deriv_ReLU(Z)
python
Copy
Edit
def deriv_ReLU(Z):
    return Z > 0
Tujuan:
Menghitung turunan dari ReLU (dibutuhkan untuk backprop).
Kalau > 0 → 1, kalau <= 0 → 0.

⚠️ 7. back_prop(...)
python
Copy
Edit
def back_prop(Z1, A1, A2, W2, Y):
    m = Y.size
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2, 2)
    dZ1 = W2.T.dot(dZ2) * deriv_ReLU(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1, 2)
    return dW1, db1, dW2, db2
Tujuan:
Backward propagation → menghitung turunan (gradient) untuk:

dW2, db2 → update output layer

dW1, db1 → update hidden layer

📌 Error:

np.sum(..., 2) → harusnya axis=1 atau axis=0, bukan angka 2.
Misalnya:

python
Copy
Edit
np.sum(dZ2, axis=1, keepdims=True)
X belum didefinisikan dalam fungsi, harus dikirim sebagai argumen.

⚠️ 8. update_params(...)
python
Copy
Edit
def update_params(W1, b1, W2, b2, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2
Tujuan:
Update bobot dan bias pakai gradient descent.

In [ ]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, alpha, iterations):
    W1, b1, W2, b2 = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 10 == 0:
            print("Iteration: ", i)
            predictions = get_predictions(A2)
            print(get_accuracy(predictions, Y))
    return W1, b1, W2, b2

In [ ]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 0.10, 500)

In [ ]:
def make_predictions(X, W1, b1, W2, b2):
    _, _, _, A2 = forward_prop(W1, b1, W2, b2, X)
    predictions = get_predictions(A2)
    return predictions

def test_prediction(index, W1, b1, W2, b2):
    current_image = X_train[:, index, None]
    prediction = make_predictions(X_train[:, index, None], W1, b1, W2, b2)
    label = Y_train[index]
    print("Prediction: ", prediction)
    print("Label: ", label)
    
    current_image = current_image.reshape((28, 28)) * 255
    plt.gray()
    plt.imshow(current_image, interpolation='nearest')
    plt.show()

In [ ]:
test_prediction(0, W1, b1, W2, b2)
test_prediction(1, W1, b1, W2, b2)
test_prediction(2, W1, b1, W2, b2)
test_prediction(3, W1, b1, W2, b2)

In [ ]:
dev_predictions = make_predictions(X_dev, W1, b1, W2, b2)
get_accuracy(dev_predictions, Y_dev)